# Logistic Regression: Classification with Probabilities

Logistic regression is one of the most fundamental and widely used algorithms for binary classification. Despite its name, it is a **classification** algorithm — not a regression algorithm. It estimates the probability that an instance belongs to a particular class using the logistic (sigmoid) function.

In this notebook, we will cover:
- Why logistic regression is called "regression" but used for classification
- The sigmoid function and how it maps outputs to probabilities
- Odds ratio, log-odds, and the decision boundary
- Maximum Likelihood Estimation (MLE) and the log-loss cost function
- Implementing logistic regression from scratch with gradient descent
- Using scikit-learn's `LogisticRegression`
- Multi-class classification (OvR vs Softmax)
- Regularization, hyperparameters, and evaluation metrics
- Real-world examples on Breast Cancer and Iris datasets
- Comparison with other classifiers

---
## 1. Why "Logistic" Regression?

The name is admittedly confusing. Logistic **regression** uses a linear model under the hood, but wraps the output through a non-linear logistic function so the result is a probability in [0, 1].

**Key insight:**
- Linear regression: $y = w^T x + b$  → unbounded continuous output
- Logistic regression: $P(y=1|x) = \sigma(w^T x + b)$  → probability bounded in [0,1]

Logistic regression belongs to the **Generalized Linear Model (GLM)** family. The "regression" part refers to the linear relationship in the log-odds space, not the final classification.

---
## 2. The Sigmoid / Logistic Function

The sigmoid function is defined as:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Properties:
- Maps any real number $z$ to a value between 0 and 1
- $\sigma(0) = 0.5$ — the decision threshold
- Symmetric around (0, 0.5)
- Differentiable everywhere (important for gradient-based optimization)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

z = np.linspace(-10, 10, 300)
sigma = sigmoid(z)

plt.figure(figsize=(8, 4))
plt.plot(z, sigma, 'b-', linewidth=2, label=r'$\sigma(z) = \frac{1}{1+e^{-z}}$')
plt.axvline(0, color='gray', linestyle='--', alpha=0.5)
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Threshold = 0.5')
plt.axhline(0, color='gray', alpha=0.3)
plt.axhline(1, color='gray', alpha=0.3)
plt.xlabel('z = wTx + b')
plt.ylabel(r'$\sigma(z)$')
plt.title('Sigmoid / Logistic Function')
plt.legend()
plt.ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

---
## 3. Odds Ratio and Log-Odds

**Odds** are a way of expressing probability ratios:

$$\text{Odds} = \frac{p}{1-p}$$

If $p=0.8$, odds are $0.8/0.2 = 4$ (4:1 in favor).

**Log-odds** (logit function) is the inverse of the sigmoid:

$$\text{logit}(p) = \ln\left(\frac{p}{1-p}\right)$$

In logistic regression, the model assumes a **linear relationship in log-odds space**:

$$\ln\left(\frac{p}{1-p}\right) = w^T x + b$$

This means each feature $x_i$ has a multiplicative effect on the odds via $e^{w_i}$:
- If $w_i > 0$: increasing $x_i$ increases the odds of class 1
- If $w_i < 0$: increasing $x_i$ decreases the odds of class 1

In [ ]:
p = np.linspace(0.01, 0.99, 100)
odds = p / (1 - p)
log_odds = np.log(odds)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(p, odds, 'g-', linewidth=2)
ax1.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
ax1.set_xlabel('Probability p')
ax1.set_ylabel('Odds = p/(1-p)')
ax1.set_title('Odds Ratio')
ax1.set_ylim(0, 20)

ax2.plot(p, log_odds, 'r-', linewidth=2)
ax2.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('Probability p')
ax2.set_ylabel('Log-Odds = ln(p/(1-p))')
ax2.set_title('Log-Odds (Logit)')

plt.tight_layout()
plt.show()

---
## 4. Decision Boundary

Logistic regression separates classes with a **linear decision boundary**:

$$w^T x + b = 0$$

- If $w^T x + b > 0$ → $\sigma(z) > 0.5$ → predict class 1
- If $w^T x + b < 0$ → $\sigma(z) < 0.5$ → predict class 0

The boundary is linear in the feature space — logistic regression is fundamentally a **linear classifier**.

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=200, n_features=2, n_redundant=0,
                           n_clusters_per_class=1, random_state=42)

def plot_decision_boundary(X, y, w, b):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                          np.linspace(y_min, y_max, 200))
    Z = sigmoid(np.dot(np.c_[xx.ravel(), yy.ravel()], w) + b)
    Z = Z.reshape(xx.shape)
    
    plt.contourf(xx, yy, Z, levels=20, cmap='RdBu', alpha=0.6)
    plt.contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=2)
    scatter = plt.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu', edgecolors='k', s=40)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.title('Decision Boundary of Logistic Regression')
    plt.colorbar(scatter, label='Class')

w = np.array([1.5, -0.8])
b = 0.0

plt.figure(figsize=(8, 6))
plot_decision_boundary(X, y, w, b)
plt.tight_layout()
plt.show()

---
## 5. Maximum Likelihood Estimation (MLE) Intuition

Instead of minimizing squared error (as in linear regression), logistic regression uses **Maximum Likelihood Estimation**.

**The idea:** Find parameters $w, b$ that maximize the probability (likelihood) of the observed data.

For a single sample $(x_i, y_i)$:
- If $y_i = 1$: likelihood = $\hat{p}_i = \sigma(w^T x_i + b)$
- If $y_i = 0$: likelihood = $1 - \hat{p}_i$

Combined: $$L_i = \hat{p}_i^{y_i} (1 - \hat{p}_i)^{1 - y_i}$$

Overall likelihood (assuming independence): $$L(w,b) = \prod_{i=1}^n L_i$$

We take the log (monotonic transformation) and negate to get the **log-loss**.

---
## 6. Cost Function: Log Loss (Binary Cross-Entropy)

The log-loss (binary cross-entropy) for a single sample is:

$$\text{Loss} = -\big[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \big]$$

For the entire dataset:

$$J(w,b) = -\frac{1}{n} \sum_{i=1}^n \big[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \big]$$

**Intuition:**
- If $y=1$ and $\hat{p}$ is close to 1 → loss ≈ 0
- If $y=1$ and $\hat{p}$ is close to 0 → loss → ∞ (heavily penalizes confident wrong predictions)

Log-loss is convex, which guarantees a single global minimum for gradient descent to find.

In [ ]:
p_hat = np.linspace(0.001, 0.999, 200)
loss_y1 = -np.log(p_hat)
loss_y0 = -np.log(1 - p_hat)

plt.figure(figsize=(8, 4))
plt.plot(p_hat, loss_y1, 'r-', label='Loss when y=1')
plt.plot(p_hat, loss_y0, 'b-', label='Loss when y=0')
plt.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('Predicted Probability $\\hat{p}$')
plt.ylabel('Loss')
plt.title('Binary Cross-Entropy (Log Loss)')
plt.legend()
plt.ylim(0, 5)
plt.tight_layout()
plt.show()

---
## 7. Implementing Logistic Regression from Scratch

We'll implement logistic regression using gradient descent:

**Gradient of log-loss w.r.t weights:**

$$\frac{\partial J}{\partial w_j} = \frac{1}{n} \sum_{i=1}^n (\hat{p}_i - y_i) \, x_{ij}$$

**Gradient w.r.t bias:**

$$\frac{\partial J}{\partial b} = \frac{1}{n} \sum_{i=1}^n (\hat{p}_i - y_i)$$

**Gradient descent update:**
$$w := w - \alpha \frac{\partial J}{\partial w}$$

In [ ]:
class LogisticRegressionScratch:
    """Logistic Regression implemented from scratch using gradient descent."""
    
    def __init__(self, lr=0.01, n_iters=1000, verbose=False):
        self.lr = lr
        self.n_iters = n_iters
        self.verbose = verbose
        self.weights = None
        self.bias = None
        self.loss_history = []
    
    def _sigmoid(self, z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -250, 250)))
    
    def _log_loss(self, y_true, y_pred):
        eps = 1e-12
        y_pred = np.clip(y_pred, eps, 1 - eps)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        
        for i in range(self.n_iters):
            linear = np.dot(X, self.weights) + self.bias
            y_pred = self._sigmoid(linear)
            
            dw = (1.0 / n_samples) * np.dot(X.T, (y_pred - y))
            db = (1.0 / n_samples) * np.sum(y_pred - y)
            
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
            loss = self._log_loss(y, y_pred)
            self.loss_history.append(loss)
            
            if self.verbose and i % 100 == 0:
                print(f'Iter {i:4d}, Loss: {loss:.6f}')
        
        return self
    
    def predict_proba(self, X):
        linear = np.dot(X, self.weights) + self.bias
        return self._sigmoid(linear)
    
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=500, n_features=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_scratch = LogisticRegressionScratch(lr=0.1, n_iters=2000, verbose=True)
model_scratch.fit(X_train_scaled, y_train)

y_pred_scratch = model_scratch.predict(X_test_scaled)
train_acc = model_scratch.score(X_train_scaled, y_train)
test_acc = model_scratch.score(X_test_scaled, y_test)

print(f'\nTrain accuracy: {train_acc:.4f}')
print(f'Test accuracy:  {test_acc:.4f}')

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(model_scratch.loss_history, 'b-', linewidth=1)
plt.xlabel('Iteration')
plt.ylabel('Log Loss')
plt.title('Training Loss over Iterations (Gradient Descent)')
plt.tight_layout()
plt.show()

---
## 8. Implementation with scikit-learn

In [ ]:
from sklearn.linear_model import LogisticRegression

sk_model = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=42)
sk_model.fit(X_train_scaled, y_train)

y_pred_sk = sk_model.predict(X_test_scaled)
y_proba_sk = sk_model.predict_proba(X_test_scaled)[:, 1]

train_acc_sk = sk_model.score(X_train_scaled, y_train)
test_acc_sk = sk_model.score(X_test_scaled, y_test)

print(f'scikit-learn LogisticRegression')
print(f'Train accuracy: {train_acc_sk:.4f}')
print(f'Test accuracy:  {test_acc_sk:.4f}')
print(f'\nWeights: {sk_model.coef_.flatten()}')
print(f'Bias:    {sk_model.intercept_[0]:.4f}')

---
## 9. Multi-Class Classification: OvR vs Softmax

Logistic regression extends to multi-class problems via two strategies:

### One-vs-Rest (OvR)
- Train $K$ binary classifiers (one for each class vs all others)
- At prediction time, pick the class with the highest confidence
- Default in `LogisticRegression(multi_class='ovr')`

### Multinomial (Softmax Regression)
- Uses the softmax function instead of sigmoid
- $P(y=c \mid x) = \frac{e^{w_c^T x + b_c}}{\sum_{j=1}^K e^{w_j^T x + b_j}}$
- Single model, probabilities sum to 1 across classes
- Set via `LogisticRegression(multi_class='multinomial', solver='lbfgs')`

**Softmax function** generalizes sigmoid to $K$ classes.

In [ ]:
def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

z_example = np.array([[2.0, 1.0, 0.1],
                       [0.5, 1.5, 0.3],
                       [0.1, 0.2, 2.0]])
probs = softmax(z_example)

print('Logits (z):')
print(z_example)
print('\nSoftmax probabilities (sum to 1 per row):')
print(probs)
print('\nRow sums:', probs.sum(axis=1))

---
## 10. Regularization in Logistic Regression

Logistic regression supports three types of regularization to prevent overfitting:

| Regularization | Penalty | Effect | Parameter |
|---|---|---|---|
| **L2** (Ridge) | $\frac{1}{2}\sum w_j^2$ | Shrinks weights toward zero (default) | `penalty='l2'` |
| **L1** (Lasso) | $\sum |w_j|$ | Drives some weights to exactly zero (feature selection) | `penalty='l1'`, `solver='saga'` |
| **ElasticNet** | $\rho \cdot L1 + (1-\rho) \cdot L2$ | Mix of both | `penalty='elasticnet'`, `l1_ratio=ρ` |

The **cost function with L2 regularization**:

$$J(w,b) = -\frac{1}{n} \sum_{i=1}^n [y_i \log \hat{p}_i + (1-y_i) \log(1 - \hat{p}_i)] + \frac{1}{C} \cdot \frac{1}{2} \sum_{j=1}^m w_j^2$$

**Note:** Parameter `C` is the **inverse** of regularization strength — smaller `C` = stronger regularization.

In [ ]:
from sklearn.model_selection import cross_val_score

C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
cv_scores = []

for C in C_values:
    model = LogisticRegression(C=C, solver='lbfgs', max_iter=1000, random_state=42)
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

plt.figure(figsize=(8, 4))
plt.semilogx(C_values, cv_scores, 'bo-', linewidth=2, markersize=8)
plt.xlabel('C (inverse regularization strength)')
plt.ylabel('Cross-Validation Accuracy')
plt.title('Effect of Regularization Strength on Accuracy')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for C, score in zip(C_values, cv_scores):
    print(f'C = {C:7.3f}  ->  CV Accuracy = {score:.4f}')

---
## 11. Key Hyperparameters

| Hyperparameter | Description | Common Values |
|---|---|---|
| `C` | Inverse regularization strength | `0.001`, `0.01`, `0.1`, `1.0`, `10` |
| `penalty` | Type of regularization | `'l1'`, `'l2'`, `'elasticnet'`, `'none'` |
| `solver` | Optimization algorithm | `'lbfgs'`, `'liblinear'`, `'saga'`, `'newton-cg'` |
| `max_iter` | Maximum iterations for solver | `100`, `500`, `1000`, `5000` |
| `multi_class` | Multi-class strategy | `'ovr'`, `'multinomial'` |

**Solver recommendations:**
- `'lbfgs'`: Default, good for small datasets, supports L2 and multinomial
- `'liblinear'`: Good for small datasets, supports L1 and L2, OvR only
- `'saga'`: Best for large datasets, supports all penalties including ElasticNet
- `'newton-cg'`: Good for multinomial, supports L2

---
## 12. Evaluation Metrics

### Confusion Matrix

| | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actual Positive** | TP (True Positive) | FN (False Negative) |
| **Actual Negative** | FP (False Positive) | TN (True Negative) |

### Derived Metrics
- **Accuracy**: $(TP + TN) / (TP + TN + FP + FN)$
- **Precision**: $TP / (TP + FP)$ — how many predicted positives are correct
- **Recall (Sensitivity)**: $TP / (TP + FN)$ — how many actual positives are found
- **F1 Score**: $2 \cdot (Precision \cdot Recall) / (Precision + Recall)$ — harmonic mean
- **ROC Curve**: TPR vs FPR at various thresholds
- **AUC**: Area under ROC curve — overall measure of separability

In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, auc, precision_recall_curve)

cm = confusion_matrix(y_test, y_pred_sk)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'])
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(y_test, y_pred_sk))

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba_sk)
roc_auc = auc(fpr, tpr)

precision, recall, _ = precision_recall_curve(y_test, y_proba_sk)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random classifier')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(recall, precision, 'r-', linewidth=2, label='Precision-Recall curve')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'AUC Score: {roc_auc:.4f}')

---
## 13. Feature Importance from Coefficients

Since logistic regression is a linear model in log-odds space, the coefficients directly tell us about feature importance:

- **Sign**: Positive → increases probability of class 1; Negative → decreases it
- **Magnitude**: Larger absolute value → stronger influence
- **Odds ratio**: $e^{w_j}$ — a one-unit increase in $x_j$ multiplies the odds of class 1 by $e^{w_j}$

In [ ]:
feature_names = [f'Feature_{i+1}' for i in range(X_train_scaled.shape[1])]
coefs = sk_model.coef_.flatten()
odds_ratios = np.exp(coefs)

colors = ['red' if c < 0 else 'green' for c in coefs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

bars1 = ax1.barh(feature_names, coefs, color=colors, edgecolor='k')
ax1.axvline(0, color='black', linewidth=0.8)
ax1.set_xlabel('Coefficient Value')
ax1.set_title('Logistic Regression Coefficients')

bars2 = ax2.barh(feature_names, odds_ratios, color='steelblue', edgecolor='k')
ax2.axvline(1, color='black', linewidth=0.8, linestyle='--')
ax2.set_xlabel('Odds Ratio (exp(coef))')
ax2.set_title('Odds Ratios')

plt.tight_layout()
plt.show()

print('Feature Importance:')
for name, coef, odds in zip(feature_names, coefs, odds_ratios):
    direction = 'increases' if coef > 0 else 'decreases'
    print(f'  {name}: coef={coef:+.4f}, odds_ratio={odds:.4f} ({direction} class 1 odds)')

---
## 14. Real-World Example: Breast Cancer Dataset (Binary Classification)

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target

print(f'Dataset shape: {X_c.shape}')
print(f'Feature names: {list(cancer.feature_names[:5])}...')
print(f'Target classes: {cancer.target_names}')
print(f'Class distribution: 0={sum(y_c==0)}, 1={sum(y_c==1)}')

X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42, stratify=y_c
)

scaler_c = StandardScaler()
X_c_train_s = scaler_c.fit_transform(X_c_train)
X_c_test_s = scaler_c.transform(X_c_test)

lr_cancer = LogisticRegression(C=1.0, solver='lbfgs', max_iter=5000, random_state=42)
lr_cancer.fit(X_c_train_s, y_c_train)

y_c_pred = lr_cancer.predict(X_c_test_s)
y_c_proba = lr_cancer.predict_proba(X_c_test_s)[:, 1]

print(f'\nTrain accuracy: {lr_cancer.score(X_c_train_s, y_c_train):.4f}')
print(f'Test accuracy:  {lr_cancer.score(X_c_test_s, y_c_test):.4f}')
print(f'ROC AUC:        {roc_auc_score(y_c_test, y_c_proba):.4f}')

In [ ]:
cm_cancer = confusion_matrix(y_c_test, y_c_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_cancer, annot=True, fmt='d', cmap='Greens', cbar=True,
            xticklabels=cancer.target_names,
            yticklabels=cancer.target_names)
plt.title('Confusion Matrix — Breast Cancer Dataset')
plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(y_c_test, y_c_pred, target_names=cancer.target_names))

In [ ]:
fpr_c, tpr_c, _ = roc_curve(y_c_test, y_c_proba)
auc_c = auc(fpr_c, tpr_c)

plt.figure(figsize=(8, 6))
plt.plot(fpr_c, tpr_c, 'g-', linewidth=2, label=f'ROC (AUC = {auc_c:.4f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Breast Cancer')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
top_k = 10
coefs_c = lr_cancer.coef_.flatten()
top_indices = np.argsort(np.abs(coefs_c))[-top_k:]

plt.figure(figsize=(8, 5))
colors_c = ['red' if coefs_c[i] < 0 else 'green' for i in top_indices]
plt.barh([cancer.feature_names[i] for i in top_indices],
         coefs_c[top_indices], color=colors_c, edgecolor='k')
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Coefficient Value')
plt.title(f'Top {top_k} Most Influential Features — Breast Cancer')
plt.tight_layout()
plt.show()

---
## 15. Real-World Example: Iris Dataset (Multi-Class with Softmax)

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

iris = load_iris()
X_i, y_i = iris.data, iris.target

print(f'Iris dataset shape: {X_i.shape}')
print(f'Classes: {iris.target_names}')

X_i_train, X_i_test, y_i_train, y_i_test = train_test_split(
    X_i, y_i, test_size=0.2, random_state=42, stratify=y_i
)

scaler_i = StandardScaler()
X_i_train_s = scaler_i.fit_transform(X_i_train)
X_i_test_s = scaler_i.transform(X_i_test)

# Softmax (multinomial) logistic regression
lr_iris = LogisticRegression(
    C=1.0, multi_class='multinomial', solver='lbfgs', max_iter=1000, random_state=42
)
lr_iris.fit(X_i_train_s, y_i_train)

y_i_pred = lr_iris.predict(X_i_test_s)
y_i_proba = lr_iris.predict_proba(X_i_test_s)

print(f'\nTrain accuracy: {lr_iris.score(X_i_train_s, y_i_train):.4f}')
print(f'Test accuracy:  {lr_iris.score(X_i_test_s, y_i_test):.4f}')
print(f'\nSample predictions (first 5):')
for i in range(5):
    print(f'  Pred: {iris.target_names[y_i_pred[i]]} '
          f'(probs: {np.round(y_i_proba[i], 3)}) '
          f'True: {iris.target_names[y_i_test[i]]}')

In [ ]:
cm_iris = confusion_matrix(y_i_test, y_i_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm_iris, annot=True, fmt='d', cmap='Purples', cbar=True,
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.title('Confusion Matrix — Iris Dataset (Softmax Regression)')
plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(y_i_test, y_i_pred, target_names=iris.target_names))

In [ ]:
# Visualize multi-class coefficients
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, class_idx in zip(axes, range(3)):
    coefs_class = lr_iris.coef_[class_idx]
    colors_i = ['red' if c < 0 else 'green' for c in coefs_class]
    ax.barh(iris.feature_names, coefs_class, color=colors_i, edgecolor='k')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Coefficient')
    ax.set_title(f'Class: {iris.target_names[class_idx]}')

plt.suptitle('Coefficients per Class — Iris (Softmax Regression)', fontsize=14)
plt.tight_layout()
plt.show()

---
## 16. Comparing Logistic Regression with Other Classifiers

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

classifiers = {
    'Logistic Regression': LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)':           SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
    'KNN (k=5)':           KNeighborsClassifier(n_neighbors=5)
}

results = {}
for name, clf in classifiers.items():
    scores = cross_val_score(clf, X_c_train_s, y_c_train, cv=5, scoring='accuracy')
    results[name] = (scores.mean(), scores.std())
    print(f'{name:22s}  CV Accuracy: {scores.mean():.4f} ± {scores.std():.4f}')

# Plot comparison
names = list(results.keys())
means = [results[n][0] for n in names]
stds = [results[n][1] for n in names]

plt.figure(figsize=(10, 5))
bars = plt.barh(names, means, xerr=stds, color='cornflowerblue', edgecolor='k', capsize=5)
plt.xlabel('Cross-Validation Accuracy')
plt.title('Classifier Comparison — Breast Cancer Dataset')
plt.xlim(0.85, 1.0)
plt.tight_layout()
plt.show()

print('\nKey Takeaways:')
print('- Logistic Regression performs competitively on linearly separable problems.')
print('- It is fast to train and highly interpretable (coefficients = feature importance).')
print('- For non-linear decision boundaries, tree-based models or kernel SVM may outperform.')

---
## 17. Pros and Cons of Logistic Regression

### Pros
- **Interpretable**: Coefficients directly tell you direction and magnitude of feature impact
- **Probabilistic**: Outputs calibrated probabilities, not just hard labels
- **Efficient**: Fast to train, scales well to large datasets with `solver='saga'`
- **Low variance**: Rarely overfits heavily (especially with regularization)
- **Well-calibrated**: Predicted probabilities are generally well-calibrated

### Cons
- **Linear decision boundary**: Cannot capture non-linear relationships without feature engineering
- **Feature engineering needed**: Requires interaction terms or polynomial features for complex patterns
- **Sensitive to outliers**: Can pull decision boundary significantly
- **Assumes independence**: Features should not be highly correlated (multicollinearity)
- **Class imbalance**: Performance degrades on severely imbalanced datasets

---
## 18. Practice Exercises

Try these exercises to deepen your understanding:

### Exercise 1: Effect of C on Coefficients
Train logistic regression models with `C = [0.001, 0.01, 0.1, 1, 10, 100]` on the Breast Cancer dataset. Plot how the coefficient values change as `C` varies. What happens to the magnitude of coefficients as regularization weakens?

### Exercise 2: Decision Boundary for 2D Synthetic Data
Generate 2D data with `make_classification(n_features=2)` and fit logistic regression with `C=0.01`, `C=1.0`, and `C=100`. Plot the decision boundary for each. How does the boundary change?

### Exercise 3: Log-Loss from Scratch
Write a function `binary_cross_entropy(y_true, y_pred)` that computes the log-loss without using scikit-learn. Compare your implementation with `sklearn.metrics.log_loss` on random arrays.

### Exercise 4: Probability Calibration
Use `sklearn.calibration.CalibrationDisplay` to plot calibration curves for logistic regression vs random forest on the Breast Cancer dataset. Which model is better calibrated?

### Exercise 5: Feature Selection via L1
Use `LogisticRegression(penalty='l1', solver='saga', C=0.1)` on the Breast Cancer dataset. How many coefficients are set to exactly zero? Which features were selected? Vary `C` and observe the sparsity pattern.

---
## Summary

In this notebook, we covered:

1. **Why logistic regression** is a classification algorithm despite the name
2. **Sigmoid function** that maps linear outputs to [0,1] probabilities
3. **Odds ratio and log-odds** — the linear relationship in log-odds space
4. **Decision boundary** — linear separation between classes
5. **MLE and log-loss** — how logistic regression optimizes parameters
6. **From-scratch implementation** with gradient descent
7. **scikit-learn implementation** with proper API usage
8. **Multi-class classification** via OvR and Softmax
9. **Regularization** — L1, L2, ElasticNet, and the role of `C`
10. **Hyperparameters** — solver, max_iter, and their trade-offs
11. **Evaluation** — confusion matrix, ROC, AUC, precision, recall, F1
12. **Feature importance** from coefficients and odds ratios
13. **Real-world examples** on Breast Cancer and Iris datasets
14. **Comparison** with decision trees, random forests, SVM, and KNN

Logistic regression is a cornerstone of machine learning — it is often the first algorithm to try for binary classification, serving as a strong baseline that is fast, interpretable, and often surprisingly effective.